# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
spaceship = spaceship.dropna()
from sklearn.model_selection import train_test_split

X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [5]:
from sklearn.preprocessing import OneHotEncoder

onehot_columns = ["Cabin", "HomePlanet", "Destination"]
drop_columns = ["PassengerId", "Name"]
bool_columns = ["CryoSleep", "VIP"]

X_train = X_train.copy()
X_test = X_test.copy()

X_train["Cabin"] = X_train["Cabin"].str[0]
X_test["Cabin"] = X_test["Cabin"].str[0]

onehot_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded_train = onehot_encoder.fit_transform(X_train[onehot_columns])
encoded_test = onehot_encoder.transform(X_test[onehot_columns])

encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=onehot_encoder.get_feature_names_out(onehot_columns),
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=onehot_encoder.get_feature_names_out(onehot_columns),
    index=X_test.index
)

X_train = pd.concat(
    [
        X_train.drop(columns=onehot_columns + drop_columns),
        encoded_train_df
    ],
    axis=1
)

X_test = pd.concat(
    [
        X_test.drop(columns=onehot_columns + drop_columns),
        encoded_test_df
    ],
    axis=1
)

X_train[bool_columns] = X_train[bool_columns].astype(int)
X_test[bool_columns] = X_test[bool_columns].astype(int)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Accuracy:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}

Best CV Accuracy:
0.7957992890112096


In [8]:
best_rf = grid_search.best_estimator_

y_pred_best_rf = best_rf.predict(X_test)

print("Test Accuracy:")
print(accuracy_score(y_test, y_pred_best_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_rf))

Test Accuracy:
0.8139183055975794

Classification Report:
              precision    recall  f1-score   support

       False       0.81      0.81      0.81       653
        True       0.82      0.82      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322


Confusion Matrix:
[[530 123]
 [123 546]]


- Evaluate your model

In [9]:
train_pred = best_rf.predict(X_train)
test_pred = best_rf.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

Train Accuracy: 0.8544663133989402
Test Accuracy: 0.8139183055975794


In [ ]:
#No critical overfitting

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

- Evaluate your model